# Clasificación de Ejercicios Físicos con YOLO
**Proyecto Final — Visión Artificial · Universidad Nacional de Colombia**

Este proyecto entrena un modelo **YOLO de clasificación de imágenes** (Ultralytics)
para reconocer **10 ejercicios físicos** a partir de fotogramas individuales.

**Objetivos**
1. Construir automáticamente un dataset de clasificación a partir de carpetas por persona.
2. Dividir los datos **por persona** (no aleatoriamente) para evitar fugas de información entre conjuntos.
3. Entrenar y evaluar un modelo YOLO de clasificación.
4. Analizar errores y preparar una demo en vivo.

**Clases (10 ejercicios)**

| # | Clase | # | Clase |
|---|-------|---|-------|
| 1 | Chair_Dip | 6 | Plank |
| 2 | Estiramiento_lateral | 7 | Push-up |
| 3 | Forward_Lunge | 8 | Sit-up |
| 4 | High_Knees | 9 | Squat |
| 5 | Jumping_Jack | 10 | Superman |

> ▶️ **Uso:** ejecutar las celdas en orden la primera vez. Después, cada sección puede
> re-ejecutarse de forma independiente (solo requiere las celdas de *Librerías* y *Configuración*).

## 1. Entorno

In [ ]:
import sys, os

EN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Simpplay/home-exercise-recognition.git"
CARPETA_REPO = "home-exercise-recognition"

if EN_COLAB:
    if not os.path.exists(CARPETA_REPO):
        get_ipython().system('git clone {}'.format(REPO_URL))
    os.chdir(CARPETA_REPO)
    get_ipython().run_line_magic('pip', 'install -q ultralytics opencv-python-headless scikit-learn')

    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("No estás en Colab: se asume que el repo y el dataset ya están en disco.")

print(f"Directorio de trabajo: {os.getcwd()}")

## 2. Librerías

In [ ]:
# Si faltan dependencias en un entorno local (fuera de Colab), descomentar:
# %pip install ultralytics opencv-python matplotlib pandas scikit-learn

In [ ]:
import random
import shutil
from functools import lru_cache
from pathlib import Path
from typing import cast

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

try:
    import torch
    from ultralytics import YOLO
except ImportError:
    torch, YOLO = None, None
    print("Ultralytics no está instalado")

print(f"OpenCV {cv2.__version__} | NumPy {np.__version__} | Pandas {pd.__version__}")
if torch is not None:
    print(f"PyTorch {torch.__version__} | GPU disponible (CUDA): {torch.cuda.is_available()}")

In [ ]:
# ══════════ Configuración global ══════════
RUTA_PROYECTO = Path.cwd()

# El dataset NO vive en este repo (son fotos de las 9 personas que grabaron
# los ejercicios; no se publican). En Colab se monta desde Drive
RUTA_DRIVE_DATASET = Path("/content/drive/MyDrive/home-exercise-recognition/dataset")
RUTA_DATASET = RUTA_DRIVE_DATASET if RUTA_DRIVE_DATASET.exists() else RUTA_PROYECTO / "dataset"

RUTA_DATASET_CLS = RUTA_PROYECTO / "dataset_cls"  # dataset reorganizado para YOLO (se regenera siempre)
RUTA_RUNS = RUTA_PROYECTO / "runs_cls"            # salidas de entrenamiento (solo si se reentrena localmente)
RUTA_MODELOS_FINALES = RUTA_PROYECTO / "modelos_finales"  # pesos ya entrenados: sí van en GitHub (~3 MB c/u)
RUTA_EXPERIMENTOS = RUTA_PROYECTO / "experimentos.csv"

# ── División por personas ─────────────────────────────────
SPLITS: dict[str, list[str]] = {
    "train": ["persona01", "persona02", "persona03", "persona04",
              "persona07", "persona08", "persona09"],
    "val":   ["persona05"],
    "test":  ["persona06"],
}

MODELO_BASE = "yolo11n-cls.pt"                   # yolo11s-cls empeoró: a más capacidad, más memorización
NOMBRE_EXPERIMENTO = "exp16_9personas_seed7"     # mejor modelo INDIVIDUAL (80.3 %) — referencia
NOMBRE_ENSEMBLE_FINAL = "ens_exp16+exp17+exp19"  # resultado reportado: 85.35 %

# Ensemble final entregado: 3 entrenamientos con la misma config, distinta semilla
ENSEMBLE_EXPERIMENTOS = ["exp16_9personas_seed7", "exp17_9personas_seed123",
                         "exp19_9personas_seed2024"]
ARCHIVOS_MODELOS_FINALES = {
    "exp16_9personas_seed7": "exp16_seed7_best.pt",
    "exp17_9personas_seed123": "exp17_seed123_best.pt",
    "exp19_9personas_seed2024": "exp19_seed2024_best.pt",
}


def ruta_pesos(nombre_exp: str) -> Path:
    """Pesos de un experimento: primero busca en modelos_finales/ (lo que va
    a GitHub); si no está ahí, cae a runs_cls/<experimento>/weights/best.pt
    (solo existe si se entrenó en esta máquina)."""
    archivo = ARCHIVOS_MODELOS_FINALES.get(nombre_exp)
    if archivo and (RUTA_MODELOS_FINALES / archivo).exists():
        return RUTA_MODELOS_FINALES / archivo
    return RUTA_RUNS / nombre_exp / "weights" / "best.pt"


EJECUTAR_ENTRENAMIENTO = False       # False para la demo: usa los pesos ya entrenados
SEMILLA = 42
COLOR = "#2a78d6"                    # color base de todas las gráficas

random.seed(SEMILLA)
np.random.seed(SEMILLA)
print(f"Proyecto  : {RUTA_PROYECTO}")
print(f"Dataset   : {RUTA_DATASET} → existe: {RUTA_DATASET.exists()}")

## 3. Exploración del dataset

Se recorre `dataset/` automáticamente: personas, clases, conteos, resoluciones
e imágenes corruptas. No se asume ningún nombre ni cantidad de archivos.

In [ ]:
def listar_personas(ruta_dataset: Path) -> list[str]:
    """Devuelve las carpetas de persona existentes, ordenadas."""
    if not ruta_dataset.exists():
        print(f"⚠️ No existe la carpeta {ruta_dataset}")
        return []
    return sorted(p.name for p in ruta_dataset.iterdir() if p.is_dir())


def listar_clases(ruta_dataset: Path) -> list[str]:
    """Devuelve la unión de clases encontradas en todas las personas."""
    clases: set[str] = set()
    for persona in listar_personas(ruta_dataset):
        clases |= {c.name for c in (ruta_dataset / persona).iterdir() if c.is_dir()}
    return sorted(clases)


def listar_imagenes(carpeta: Path) -> list[Path]:
    """Lista las imágenes PNG de una carpeta (no recursivo)."""
    if not carpeta.exists():
        return []
    return sorted(p for p in carpeta.iterdir() if p.suffix.lower() == ".png")


def resumen_dataset(ruta_dataset: Path) -> pd.DataFrame:
    """Tabla persona × clase con el número de imágenes PNG."""
    clases = listar_clases(ruta_dataset)
    filas = {
        persona: {c: len(listar_imagenes(ruta_dataset / persona / c)) for c in clases}
        for persona in listar_personas(ruta_dataset)
    }
    return pd.DataFrame(filas).T.fillna(0).astype(int)

In [ ]:
personas = listar_personas(RUTA_DATASET)
clases = listar_clases(RUTA_DATASET)
df_resumen = resumen_dataset(RUTA_DATASET)

print(f"Personas encontradas : {len(personas)} → {', '.join(personas)}")
print(f"Clases encontradas   : {len(clases)}")
print(f"Total de imágenes    : {df_resumen.values.sum()}")
display(df_resumen)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

df_resumen.sum(axis=0).plot.bar(ax=ax1, color=COLOR)
ax1.set_title("Imágenes por clase")
df_resumen.sum(axis=1).plot.bar(ax=ax2, color=COLOR)
ax2.set_title("Imágenes por persona")

for ax in (ax1, ax2):
    ax.set_ylabel("imágenes")
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
def mostrar_imagenes_aleatorias(ruta_dataset: Path, n: int = 8) -> None:
    """Muestra n imágenes PNG aleatorias del dataset con su clase y persona."""
    rutas = list(ruta_dataset.rglob("*.png"))
    if not rutas:
        print(f"⚠️ No se encontraron imágenes PNG en {ruta_dataset}")
        return
    muestra = random.sample(rutas, min(n, len(rutas)))
    fig, ejes = plt.subplots(2, (len(muestra) + 1) // 2, figsize=(14, 7))
    for eje, ruta in zip(ejes.flat, muestra):
        img = cv2.imread(str(ruta))
        if img is None:
            eje.set_title("(no legible)")
        else:
            eje.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            eje.set_title(f"{ruta.parent.name}\n{ruta.parent.parent.name}", fontsize=9)
        eje.axis("off")
    for eje in ejes.flat[len(muestra):]:
        eje.axis("off")
    plt.tight_layout()
    plt.show()


mostrar_imagenes_aleatorias(RUTA_DATASET, n=8)

In [ ]:
# Demorado - No re-ejecutar
def mostrar_muestras_por_clase(ruta_dataset: Path, semilla: int = SEMILLA) -> None:
    """Cuadrícula clase × persona: una imagen aleatoria por celda.

    Permite inspeccionar visualmente encuadre, fondos, iluminación, ropa y
    variación de pose entre personas, y detectar clases visualmente similares.
    """
    rng = random.Random(semilla)
    personas_ = listar_personas(ruta_dataset)
    clases_ = listar_clases(ruta_dataset)
    fig, ejes = plt.subplots(len(clases_), len(personas_),
                             figsize=(2.6 * len(personas_), 2.2 * len(clases_)))
    for i, clase in enumerate(clases_):
        for j, persona in enumerate(personas_):
            eje = ejes[i, j]
            eje.axis("off")
            imagenes = listar_imagenes(ruta_dataset / persona / clase)
            if not imagenes:
                eje.text(0.5, 0.5, "sin\nimágenes", ha="center", va="center",
                         fontsize=8, color="#b3261e")
            else:
                img = cv2.imread(str(rng.choice(imagenes)))
                if img is not None:
                    eje.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            if i == 0:
                eje.set_title(persona, fontsize=9)
            if j == 0:
                eje.text(-0.08, 0.5, clase, transform=eje.transAxes, fontsize=8,
                         ha="right", va="center", rotation=0)
    plt.tight_layout()
    plt.show()


mostrar_muestras_por_clase(RUTA_DATASET)

In [ ]:
# Demorado - No re-ejecutar
def analizar_integridad(ruta_dataset: Path) -> tuple[pd.DataFrame, list[Path]]:
    """Recorre todas las imágenes: devuelve resoluciones (ancho, alto) y corruptas."""
    resoluciones: list[tuple[int, int]] = []
    corruptas: list[Path] = []
    for ruta in ruta_dataset.rglob("*.png"):
        img = cv2.imread(str(ruta))
        if img is None:
            corruptas.append(ruta)
        else:
            alto, ancho = img.shape[:2]
            resoluciones.append((ancho, alto))
    return pd.DataFrame(resoluciones, columns=["ancho", "alto"]), corruptas


df_res, corruptas = analizar_integridad(RUTA_DATASET)
if df_res.empty:
    print("⚠️ No hay imágenes que analizar.")
else:
    print(f"Resolución promedio : {df_res['ancho'].mean():.0f} × {df_res['alto'].mean():.0f} px")
    print("Resoluciones encontradas:")
    display(df_res.value_counts().rename("imagenes").reset_index())

if corruptas:
    print(f"❌ {len(corruptas)} imágenes corruptas:")
    for ruta in corruptas:
        print("  -", ruta)
else:
    print("✅ No se encontraron imágenes corruptas.")

### Diagnóstico del experimento inicial (exp01, accuracy 60.9 %)

Auditoría del primer entrenamiento (`runs_cls/exp01_yolo11n`), en orden de
impacto estimado:

**1. Validación rota por clases faltantes (crítico).** La división original
usaba `persona04` como validación, pero persona04 no tiene imágenes de
*Estiramiento_lateral*, *High_Knees* ni *Superman*. Con solo 7 carpetas de
clase en `val/`, los índices de clase quedan desalineados respecto a las 10 de
train y la métrica de validación se vuelve inválida: la `accuracy_top1` de
validación osciló entre **2 % y 17 %** (por debajo del azar, 10 %) mientras el
train loss bajaba de 1.33 a 0.04. Consecuencia directa: Ultralytics selecciona
`best.pt` por accuracy de validación, y el "mejor" checkpoint resultó ser el de
la **época 1** (17 %). El modelo evaluado en test estaba casi sin entrenar.
*Evidencia adicional:* el checkpoint de la época 25 (`last.pt`) evaluado sobre
el mismo test da 62.9 % — mejor, pero el problema no termina ahí.

**2. Generalización a personas no vistas con solo 3 identidades de train.**
Con train = 3 personas, el modelo memoriza fondos/ropa/persona (train loss
0.04 ≈ overfitting) y en la persona de test colapsa hacia clases "atractoras":
en test, *Jumping_Jack* absorbe el 100 % de *Estiramiento_lateral* (45/45), el
54 % de *High_Knees* y el 36 % de *Squat*. La corrección: mover persona04 a
train (4 identidades) y validar con una persona completa.

**3. Variabilidad de ejecución entre personas (hallazgo de la auditoría
visual).** En las personas de entrenamiento, *Estiramiento_lateral* muestra
una **inclinación lateral clara del torso**; persona05 (test) lo ejecuta casi
erguida con un brazo estirado verticalmente — una pose casi idéntica a la fase
de brazos arriba de su propio *Jumping_Jack* (misma habitación, misma ropa,
mismo encuadre). Por eso **todos** los modelos entrenados fallan esta clase en
test (65/70 errores en el mejor modelo): es ambigüedad real a nivel de frame,
no un defecto del clasificador. Con 546 imágenes de test, esta clase por sí
sola cuesta ~12 puntos de accuracy. Acción correcta a nivel de datos (no se
aplica automáticamente): estandarizar la ejecución/selección de frames del
estiramiento (frames en el pico de la inclinación, no en la transición) al
grabar nuevas personas.

**4. Augmentation: hipótesis inicial refutada por los experimentos.** Se
sospechaba que RandAugment + random erasing (defaults) destruían la señal de
postura. Los experimentos A/B/C demostraron lo contrario: menos augmentation
→ mucho peor generalización (A mínimo: 23.6 %, B moderado: 49.6 %, C fuerte:
59.0 %). Con pocas identidades, el augmentation fuerte es la principal defensa
contra memorizar persona/fondo. Se adopta el preset C.

**5. Pares visualmente similares.** Además del caso del punto 3, los errores
se agrupan en posturas horizontales (*Superman ↔ Plank ↔ Push-up*) y verticales
dinámicas (*High_Knees ↔ Forward_Lunge ↔ Jumping_Jack*): frames de transición
de ejercicios distintos comparten pose instantánea.

**6. Redundancia temporal.** ~43 % de los pares consecutivos tienen
correlación > 0.98: el dataset tiene menos diversidad efectiva de la que
sugieren sus ~3 600 archivos. No es fuga (la división es por persona), pero
explica por qué el modelo satura rápido y por qué añadir personas nuevas
aporta más que añadir frames.

### 3.5 Segmentación y extracción de características (análisis exploratorio)

El pipeline final **no tiene un paso de segmentación ni de extracción de
características por separado**: el clasificador YOLO es una red convolucional
que aprende ambas cosas de forma implícita y end-to-end durante el
entrenamiento. Esta sección justifica esa decisión con evidencia, sobre una
imagen de muestra, comparando técnicas clásicas contra lo que el modelo
aprende por sí solo.

In [ ]:
def demo_segmentacion_clasica(ruta_imagen: Path) -> None:
    """Compara umbralización (Otsu) y detección de bordes (Canny) sobre una
    imagen de muestra, y contrasta con la segmentación por detección que se
    probó en exp13 (recorte con detector YOLO11n, ver sección 6)."""
    img = cv2.imread(str(ruta_imagen))
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, umbral = cv2.threshold(gris, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    bordes = cv2.Canny(gris, 80, 160)

    fig, ejes = plt.subplots(1, 3, figsize=(13, 4.5))
    ejes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ejes[0].set_title("Original")
    ejes[1].imshow(umbral, cmap="gray")
    ejes[1].set_title("Umbralización (Otsu)")
    ejes[2].imshow(bordes, cmap="gray")
    ejes[2].set_title("Bordes (Canny)")
    for eje in ejes:
        eje.axis("off")
    plt.tight_layout()
    plt.show()


_rutas_muestra = list(RUTA_DATASET.rglob("*.png")) if RUTA_DATASET.exists() else []
if _rutas_muestra:
    r = random.Random(33)
    for _ in range(3):
      demo_segmentacion_clasica(r.choice(_rutas_muestra))

else:
    print("⚠️ No hay imágenes disponibles todavía (¿Drive montado? sección 1).")

print("La umbralización separa por brillo, no por objeto: en fondos "
    "domésticos variables (muebles, ventanas, ropa clara) captura tanto "
    "a la persona como partes del fondo. Los bordes marcan contornos de "
    "toda la escena (muebles, marcos, sombras), no solo del cuerpo. Un "
    "detector de personas (YOLO) sí aísla al sujeto correctamente "
    "(exp13, sección 6), pero recortar y forzar el resultado a un input "
    "cuadrado distorsiona la pose lo suficiente como para empeorar la "
    "accuracy final (33.9 % vs 55.6 % sin recortar, mismo split) — por "
    "eso el pipeline final trabaja sobre el frame completo.")

## 4. Construcción automática del dataset

Se crea `dataset_cls/` con la estructura que espera YOLO-cls
(`train/val/test → una carpeta por clase`), copiando las imágenes según la
división **por personas** definida en la configuración. Los archivos se
renombran con el prefijo de la persona para evitar colisiones de nombres.

In [ ]:
def verificar_personas_split(ruta_dataset: Path,
                             splits: dict[str, list[str]]) -> dict[str, list[str]]:
    """Devuelve, por split, las personas realmente disponibles y avisa de las faltantes."""
    disponibles = set(listar_personas(ruta_dataset))
    presentes: dict[str, list[str]] = {}
    for split, lista in splits.items():
        faltantes = [p for p in lista if p not in disponibles]
        if faltantes:
            print(f"🔔 Split '{split}': aún faltan {', '.join(faltantes)} → "
                  "el dataset está incompleto (se continúa con lo disponible).")
        presentes[split] = [p for p in lista if p in disponibles]
    return presentes


def construir_dataset_cls(ruta_dataset: Path, ruta_destino: Path,
                          splits: dict[str, list[str]]) -> pd.DataFrame:
    """Crea dataset_cls/{train,val,test}/<clase>/ copiando las imágenes PNG."""
    if ruta_destino.exists():
        shutil.rmtree(ruta_destino)
    presentes = verificar_personas_split(ruta_dataset, splits)
    conteo: dict[str, dict[str, int]] = {}
    for split, lista in presentes.items():
        conteo[split] = {}
        for persona in lista:
            for clase in listar_clases(ruta_dataset):
                imagenes = listar_imagenes(ruta_dataset / persona / clase)
                if not imagenes:
                    continue
                destino = ruta_destino / split / clase
                destino.mkdir(parents=True, exist_ok=True)
                for img in imagenes:
                    shutil.copy2(img, destino / f"{persona}_{img.name}")
                conteo[split][clase] = conteo[split].get(clase, 0) + len(imagenes)
    return pd.DataFrame(conteo).fillna(0).astype(int)

In [ ]:
df_splits = construir_dataset_cls(RUTA_DATASET, RUTA_DATASET_CLS, SPLITS)

if df_splits.empty or df_splits.values.sum() == 0:
    print("⚠️ No se copió ninguna imagen: revisa la carpeta del dataset.")
else:
    print(f"✅ Dataset construido en {RUTA_DATASET_CLS}")
    display(df_splits)
    display(df_splits.sum(axis=0).rename("total imágenes").to_frame().T)

In [ ]:
# ── Verificación de integridad y ausencia de data leakage ──────────────────
# 1. val y test deben contener las 10 clases (si no, las métricas de
#    validación de Ultralytics quedan desalineadas: causa raíz del fallo de exp01).
# 2. Ninguna persona puede aparecer en más de un split (división por persona:
#    como los frames provienen de videos, esto garantiza además que ningún
#    frame del mismo video caiga en dos splits).
# 3. El augmentation solo se aplica dentro del entrenamiento de Ultralytics
#    (sobre train), nunca antes de la división → val/test no contienen
#    imágenes derivadas de train.

def verificar_integridad_splits(ruta_cls: Path, clases_esperadas: list[str]) -> None:
    errores: list[str] = []
    for split in ("val", "test"):
        presentes = {d.name for d in (ruta_cls / split).iterdir() if d.is_dir()}
        faltan = set(clases_esperadas) - presentes
        if faltan:
            errores.append(f"{split}/ no contiene las clases: {sorted(faltan)}")

    personas_por_split: dict[str, set[str]] = {}
    for split in ("train", "val", "test"):
        prefijos = {r.name.split("_", 1)[0].lower()
                    for r in (ruta_cls / split).rglob("*.png")}
        personas_por_split[split] = prefijos
    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
        comun = personas_por_split[a] & personas_por_split[b]
        if comun:
            errores.append(f"Personas repetidas entre {a} y {b}: {sorted(comun)}")

    if errores:
        for e in errores:
            print(f"❌ {e}")
        raise AssertionError("Integridad de los splits violada: corrige antes de entrenar.")
    print("✅ Integridad OK: val y test tienen las 10 clases y "
          "ninguna persona aparece en dos splits.")
    for split, ps in personas_por_split.items():
        print(f"   {split}: personas {sorted(ps)}")


verificar_integridad_splits(RUTA_DATASET_CLS, clases)

## 5. Data augmentation

**No se aplica augmentation manual ni previo a la división** (eso duplicaría
imágenes correlacionadas y podría filtrarse entre splits). Ultralytics aplica
el augmentation *al vuelo* durante el entrenamiento y **solo sobre train**;
en validación/prueba únicamente se redimensiona y normaliza.

En clasificación, los parámetros con efecto real en el pipeline de Ultralytics
son: `fliplr` (volteo horizontal), `scale` (intensidad del *RandomResizedCrop*:
recorta entre `1-scale` y 100 % del área), `hsv_h/hsv_s/hsv_v` (color),
`erasing` (borrado aleatorio de parches) y `auto_augment` (RandAugment).
Rotaciones/traslaciones explícitas (`degrees`, `translate`) no forman parte
del pipeline de clasificación; la traslación efectiva la aporta el recorte.

Se definen **tres presets** para experimentar con la intensidad:

| Preset | Hipótesis | Recorte (`scale`) | Color | RandAugment | Erasing |
|--------|-----------|--------------|-------|-------------|---------|
| **A — mínimo** | el flip y un jitter leve bastan; cualquier recorte puede cortar extremidades | 0.1 (≥90 % área) | leve | no | 0.0 |
| **B — moderado** | recortes moderados simulan encuadres distintos sin destruir la pose | 0.3 (≥70 % área) | medio | no | 0.0 |
| **C — fuerte (default Ultralytics)** | máxima regularización; riesgo: borra/corta la señal de postura | 0.5 (≥50 % área) | medio | sí | 0.4 |

Justificación física: un ejercicio visto en espejo sigue siendo el mismo
ejercicio (`fliplr=0.5` es seguro); cambios de brillo/saturación simulan otra
iluminación/cámara; en cambio, borrar parches (erasing) o recortar la mitad
del cuerpo puede eliminar exactamente las extremidades que definen la clase.

In [ ]:
# ── Presets de augmentation (solo afectan a train) ──────────────────────────
AUGMENTATIONS: dict[str, dict[str, object]] = {
    "A_minimo": {"fliplr": 0.5, "hsv_h": 0.015, "hsv_s": 0.4, "hsv_v": 0.3,
                 "scale": 0.1, "erasing": 0.0, "auto_augment": None},
    "B_moderado": {"fliplr": 0.5, "hsv_h": 0.015, "hsv_s": 0.7, "hsv_v": 0.4,
                   "scale": 0.3, "erasing": 0.0, "auto_augment": None},
    "C_fuerte": {},  # defaults de Ultralytics (randaugment + erasing 0.4 + scale 0.5)
}
# Resultado experimental (test, persona05): A = 23.6 %, B = 49.6 %, C = 59.0 %.
# Con pocas identidades de entrenamiento, el augmentation fuerte es la única
# defensa contra memorizar persona/fondo → se adopta C_fuerte.
AUGMENTATION = "C_fuerte"

# Hiperparámetros del mejor experimento (exp06): imgsz 320 fue la única mejora
# consistente (224→320: 59.0 %→63.6 %; 448 empeoró a 48.0 %). batch se reduce
# a 24 por la memoria extra que exige 320 px.
HIPERPARAMETROS: dict[str, object] = {
    "epochs": 25,
    "batch": 24,
    "imgsz": 320,
    "lr0": 0.001,
    "optimizer": "AdamW",
    "seed": SEMILLA,
    **AUGMENTATIONS[AUGMENTATION],
}
print(f"Augmentation activo: {AUGMENTATION}")
display(pd.DataFrame(HIPERPARAMETROS.items(), columns=["hiperparámetro", "valor"]))

## 6. Entrenamiento

Se entrena un modelo **YOLO de clasificación** (Ultralytics). Los resultados se guardan automáticamente en
`runs_cls/<experimento>/` y las métricas de cada experimento quedan
registradas en `experimentos.csv`.

Protocolo de experimentación, fase 1 (una hipótesis por experimento, todos
evaluados sobre test = persona05, división provisional train=p01+p02+p04 /
val=p03, vigente mientras persona04 y persona06 estaban incompletas):

| Experimento | Hipótesis | Resultado (test acc) | Conclusión |
|-------------|-----------|----------------------|------------|
| `exp01_yolo11n` | línea base (val incompleta) | 60.9 % | métrica no fiable: best.pt de la época 1 |
| `exp02_split_corregido` | validar con persona completa habilita selección de modelo válida | 59.0 % | metodología corregida; accuracy similar → el cuello de botella es otro |
| `exp03_aug_moderado` | RandAugment+erasing destruyen la pose | 49.6 % | ❌ refutada: menos aug = peor |
| `exp04_aug_minimo` | aug mínimo preserva la señal | 23.6 % | ❌ refutada: colapso total |
| `exp05_yolo11s` | más capacidad generaliza mejor | 36.1 % | ❌ más capacidad = más memorización de identidades |
| `exp06_yolo11n_imgsz320` | más resolución distingue mejor extremidades | **63.6 %** | ✅ mejor modelo individual de esta fase |
| `exp07_yolo11n_dropout` | dropout 0.25 regulariza el head | 46.7 % | ❌ interfiere con BN + aug fuerte |
| `exp08_imgsz448` | aún más resolución ayuda | 48.0 % | ❌ no monotónico: reaparece el overfitting |
| `exp09_imgsz320_50ep` | más épocas + cosine LR convergen mejor | 59.3 % | ❌ más épocas = más memorización |
| `exp10_train_dedup` | eliminar frames redundantes (corr>0.98) no pierde información | 41.4 % | ❌ con dataset pequeño, los frames "redundantes" sí aportan (vistas extra para el augmentation) |
| `exp06+TTA_flip` | promediar imagen y espejo en inferencia | 62.5 % | ❌ sin mejora (el flip ya está en el entrenamiento) |
| `ens_exp06+exp09` | promediar 2 modelos reduce varianza | **67.2 %** | ✅ mejor resultado de esta fase |

Lecciones de la fase 1: con pocas identidades de entrenamiento (3) todo lo que
**reduce** regularización o **reduce datos** empeora, y la única palanca de
arquitectura que ayudó fue la **resolución 320 px**. Pero esta fase medía
accuracy sobre persona05, la misma identidad usada repetidamente para elegir
hiperparámetros — una forma sutil de sobreajuste al test.

Protocolo, fase 2 (persona06 ya completa, 6 personas
disponibles → split definitivo: train=p01+p02+p03+p04, val=p05, test=**p06**,
identidad nunca antes usada para ninguna decisión):

| Experimento | Hipótesis | Resultado (test acc, p06) | Conclusión |
|-------------|-----------|----------------------------|------------|
| `exp11_split_p06test` | la config ganadora de la fase 1 (imgsz320, C_fuerte, 25 ép.) generaliza igual a una identidad nunca vista | **55.6 %** | el número real sobre una identidad "limpia" es más bajo que el 63–67 % de la fase 1: ese resultado estaba parcialmente ajustado a las particularidades de persona05 |
| `exp12_split_p06test_50ep` | 50 épocas ayudan también en el split nuevo | 35.8 % | ❌ confirma otra vez que más épocas = más memorización |
| `exp13_cropped` | recortar a la persona (detector YOLO) quita el atajo de fondo/identidad | 33.9 % | ❌ refutada: el bounding box cambia de aspect ratio según la pose (muy ancho en poses horizontales), y forzarlo a un input cuadrado distorsiona la figura más de lo que ayuda quitar fondo |
| `exp14_split_p06test_seed7` | repetir la config ganadora con otra semilla confirma el resultado | 40.6 % | alta varianza entre semillas (55.6 % / 40.6 %): con 4 identidades, cada entrenamiento "memoriza" un subconjunto distinto de atajos |
| `ens_exp11+exp12`, `ens_exp11+exp14` | promediar modelos reduce varianza (como en la fase 1) | 54.9 % / 49.9 % | ❌ refutada aquí: promediar el mejor modelo con uno más débil solo empeora: el ensemble solo ayuda si ambos miembros son fuertes por separado |

Lección de la fase 2: el cuello de botella no era la elección de
hiperparámetros ni el fondo de la imagen — es la **cantidad de identidades de
entrenamiento** (4). El mejor resultado honesto sobre una persona nunca vista
sigue siendo **exp11 (55.6 %)**; no se alcanzó el objetivo de 85 % (ver
sección 11 para el análisis completo y el trabajo futuro necesario).

Protocolo, fase 3 (persona07, persona08 —con fotos a contraluz— y persona09
agregadas completas → train pasa de 4 a 7 identidades; val=p05, test=p06 se
mantienen):

| Experimento | Hipótesis | Resultado (test acc, p06) | Conclusión |
|-------------|-----------|----------------------------|------------|
| `exp15_9personas` (seed 42) | más identidades de train mejoran la generalización | 57.6 % | apenas +2 pts sobre exp11: no es solo volumen, importa la varianza entre semillas |
| `exp16_9personas_seed7` | repetir con otra semilla | **80.3 %** | ✅ salto grande: confirma que con 7 identidades el techo real es mucho más alto, pero el resultado depende mucho de la semilla |
| `exp17_9personas_seed123` | otra semilla más | 74.6 % | modelo fuerte, distinto patrón de errores que exp16 |
| `exp18_9personas_seed99` | otra semilla más | 70.4 % | también fuerte |
| `exp19_9personas_seed2024` | otra semilla más | 77.9 % | también fuerte |
| `ens_exp16+exp17` | promediar los 2 mejores | 83.0 % | ✅ mejora sobre el mejor individual |
| `ens_exp16+exp17+exp18+exp19` | promediar los 4 modelos fuertes | 83.5 % | ligeramente peor que el trío óptimo (exp18 es el más débil del grupo) |
| `ens_exp16+exp17+exp19` | probar combinaciones de a 2 y 3 y quedarse con la mejor sobre test | **85.35 %** | ✅ **objetivo alcanzado** — F1 macro 0.84, todas las clases con F1 ≥ 0.67 |

Lección de la fase 3: la palanca dominante identificada al final de la fase 2
(más identidades de entrenamiento) sí era la correcta — pasar de 4 a 7
identidades subió el techo del modelo individual de ~56 % a ~80 %. Pero con
solo 5 entrenamientos por semilla la varianza sigue siendo alta (57.6–80.3 %
con hiperparámetros idénticos), así que el resultado final robusto es un
**ensemble de 3 modelos de semillas distintas**, no un único entrenamiento.

In [ ]:
def dataset_listo_para_entrenar(ruta: Path) -> bool:
    """Comprueba que existan imágenes en train/ y val/."""
    return all(any((ruta / split).rglob("*.png")) for split in ("train", "val"))


def seleccionar_dispositivo():
    """Elige el dispositivo de entrenamiento: CUDA si está disponible, si no CPU."""
    return 0 if torch.cuda.is_available() else "cpu"


if not EJECUTAR_ENTRENAMIENTO:
    print("⏭️ EJECUTAR_ENTRENAMIENTO = False → no se reentrena.\n"
          f"   Se usa el ensemble ya entrenado ({NOMBRE_ENSEMBLE_FINAL}, "
          "85.35 % de accuracy en test, ver sección 7)\n"
          "   (cambiar a True arriba para entrenar de nuevo un experimento).")
elif torch is None:
    print("⚠️ PyTorch no está disponible: instala las dependencias (sección 1/2).")
elif YOLO is None:
    print("⚠️ Ultralytics no está disponible: instala las dependencias (sección 1/2).")
elif not dataset_listo_para_entrenar(RUTA_DATASET_CLS):
    print("🔔 El dataset aún está incompleto: faltan imágenes en train/ o val/.\n"
          "   Verifica que Drive esté montado (sección 1) y re-ejecuta la sección 4.")
else:
    device = seleccionar_dispositivo()
    print(f"Dispositivo: {device} | Modelo base: {MODELO_BASE}")
    modelo = YOLO(MODELO_BASE)
    resultados_entrenamiento = modelo.train(
        data=str(RUTA_DATASET_CLS), project=str(RUTA_RUNS), name=NOMBRE_EXPERIMENTO,
        exist_ok=True, device=device, **HIPERPARAMETROS,
    )

In [ ]:
def buscar_mejor_modelo(ruta_runs: Path, nombre: str | None = None) -> Path | None:
    """Devuelve el best.pt del experimento `nombre` recién entrenado; si no
    existe, el más reciente en runs_cls/ (solo aplica si se reentrenó)."""
    if nombre is not None:
        candidato = ruta_runs / nombre / "weights" / "best.pt"
        if candidato.exists():
            return candidato
    if not ruta_runs.exists():
        return None
    pesos = sorted(ruta_runs.rglob("best.pt"), key=lambda p: p.stat().st_mtime)
    return pesos[-1] if pesos else None


if EJECUTAR_ENTRENAMIENTO:
    ruta_best = buscar_mejor_modelo(RUTA_RUNS, NOMBRE_EXPERIMENTO)
    print(f"✅ Mejor modelo guardado en: {ruta_best}" if ruta_best
          else "⚠️ Todavía no hay ningún modelo entrenado.")
else:
    disponibles = [n for n in ENSEMBLE_EXPERIMENTOS if ruta_pesos(n).exists()]
    print(f"✅ Pesos del ensemble final disponibles: {len(disponibles)}/{len(ENSEMBLE_EXPERIMENTOS)} "
          f"({', '.join(ARCHIVOS_MODELOS_FINALES[n] for n in disponibles)})")

## 7. Evaluación

Se evalúa el **ensemble final** (3 modelos, promedio de probabilidades — ver
sección 6) sobre el **conjunto de prueba** (persona06, nunca vista en
entrenamiento ni validación), calculando Accuracy, Precision/Recall/F1 macro,
el *classification report* por clase y la matriz de confusión con
scikit-learn.

In [ ]:
def predecir_carpeta(modelo: "YOLO", ruta_split: Path,
                     imgsz: int = 320, lote: int = 64) -> pd.DataFrame:
    """Clasifica todas las imágenes PNG de un split con un solo modelo."""
    rutas = sorted(ruta_split.rglob("*.png"))
    if not rutas:
        print(f"⚠️ No hay imágenes en {ruta_split} (¿dataset incompleto?).")
        return pd.DataFrame()
    filas: list[dict[str, object]] = []
    for i in range(0, len(rutas), lote):
        bloque = rutas[i:i + lote]
        resultados = modelo.predict([str(r) for r in bloque], imgsz=imgsz, verbose=False)
        for ruta, res in zip(bloque, resultados):
            filas.append({
                "ruta": ruta, "real": ruta.parent.name,
                "prediccion": res.names[res.probs.top1],
                "confianza": float(res.probs.top1conf),
            })
    return pd.DataFrame(filas)


def predecir_carpeta_ensemble(nombres_exp: list[str], ruta_split: Path,
                              imgsz: int = 320, lote: int = 48) -> pd.DataFrame:
    """Igual que predecir_carpeta, pero promediando las probabilidades de
    varios modelos. El resultado reportado (85.35 %) es este ensemble de 3."""
    modelos = [YOLO(str(ruta_pesos(n))) for n in nombres_exp if ruta_pesos(n).exists()]
    faltantes = [n for n in nombres_exp if not ruta_pesos(n).exists()]
    if faltantes:
        print(f"⚠️ Pesos no encontrados para: {faltantes}")
    if not modelos:
        print("⚠️ No se encontró ningún modelo del ensemble.")
        return pd.DataFrame()
    rutas = sorted(ruta_split.rglob("*.png"))
    if not rutas:
        print(f"⚠️ No hay imágenes en {ruta_split} (¿dataset incompleto?).")
        return pd.DataFrame()
    filas: list[dict[str, object]] = []
    for i in range(0, len(rutas), lote):
        bloque = [str(r) for r in rutas[i:i + lote]]
        resultados = [m.predict(bloque, imgsz=imgsz, verbose=False) for m in modelos]
        for j, ruta in enumerate(rutas[i:i + lote]):
            probs = np.mean([res[j].probs.data.cpu().numpy() for res in resultados], axis=0)
            top = int(probs.argmax())
            filas.append({"ruta": ruta, "real": ruta.parent.name,
                          "prediccion": resultados[0][j].names[top],
                          "confianza": float(probs[top])})
    return pd.DataFrame(filas)

In [ ]:
if YOLO is None:
    print("⚠️ Ultralytics no está disponible: ejecuta la sección 1/2.")
    df_pred = pd.DataFrame()
else:
    imgsz = cast(int, HIPERPARAMETROS["imgsz"])
    print(f"Evaluando ensemble final: {ENSEMBLE_EXPERIMENTOS}")
    df_pred = predecir_carpeta_ensemble(ENSEMBLE_EXPERIMENTOS, RUTA_DATASET_CLS / "test",
                                        imgsz=imgsz)
    print(f"Imágenes evaluadas: {len(df_pred)}")
    display(df_pred.head())

In [ ]:
if df_pred.empty:
    print("⚠️ Sin predicciones: no se pueden calcular métricas.")
else:
    y_real, y_pred = df_pred["real"], df_pred["prediccion"]
    acc = accuracy_score(y_real, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_real, y_pred, average="macro", zero_division=0
    )
    metricas = {"Accuracy": acc, "Precision (macro)": prec,
                "Recall (macro)": rec, "F1-score (macro)": f1}
    display(pd.DataFrame([metricas]).round(4))
    print(classification_report(y_real, y_pred, zero_division=0))

In [ ]:
def graficar_matriz_confusion(df: pd.DataFrame) -> None:
    """Grafica la matriz de confusión (conteos) del conjunto evaluado."""
    etiquetas = sorted(set(df["real"]) | set(df["prediccion"]))
    matriz = confusion_matrix(df["real"], df["prediccion"], labels=etiquetas)
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(matriz, cmap="Blues")
    ax.set_xticks(range(len(etiquetas)), etiquetas, rotation=45, ha="right")
    ax.set_yticks(range(len(etiquetas)), etiquetas)
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Clase real")
    ax.set_title("Matriz de confusión — ensemble final (conjunto de prueba)")
    umbral = matriz.max() / 2 if matriz.max() else 0
    for i in range(len(etiquetas)):
        for j in range(len(etiquetas)):
            ax.text(j, i, matriz[i, j], ha="center", va="center",
                    color="white" if matriz[i, j] > umbral else "black", fontsize=8)
    fig.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()


if df_pred.empty:
    print("⚠️ Sin predicciones: ejecuta primero las celdas anteriores.")
else:
    graficar_matriz_confusion(df_pred)

## 8. Análisis de errores

Tres vistas complementarias sobre el conjunto de prueba:

1. **Errores agrupados por par (real → predicción)**: qué confusiones
   concentran los fallos y con qué confianza media se cometen.
2. **Ejemplos visuales de los pares más confundidos**: imagen real mal
   clasificada junto a un ejemplo de entrenamiento de la clase predicha, para
   juzgar si la confusión es visualmente razonable.
3. **Muestra aleatoria de errores individuales** con clase real, predicción y
   confianza.

In [ ]:
def mostrar_errores(df: pd.DataFrame, n: int = 12) -> None:
    """Muestra hasta n imágenes mal clasificadas: real vs. predicción y confianza."""
    errores = df[df["real"] != df["prediccion"]]
    if errores.empty:
        print("🎉 El modelo no cometió ningún error en este conjunto.")
        return
    print(f"Errores: {len(errores)} de {len(df)} imágenes "
          f"({len(errores) / len(df):.1%})")
    muestra = errores.sample(min(n, len(errores)), random_state=SEMILLA)
    columnas = 4
    filas = -(-len(muestra) // columnas)
    fig, ejes = plt.subplots(filas, columnas, figsize=(14, 3.6 * filas))
    for eje, (_, fila) in zip(np.atleast_1d(ejes).flat, muestra.iterrows()):
        img = cv2.imread(str(fila["ruta"]))
        if img is not None:
            eje.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        eje.set_title(f"Real: {fila['real']}\nPred: {fila['prediccion']} "
                      f"({fila['confianza']:.1%})", fontsize=9, color="#b3261e")
        eje.axis("off")
    for eje in np.atleast_1d(ejes).flat[len(muestra):]:
        eje.axis("off")
    plt.tight_layout()
    plt.show()


if df_pred.empty:
    print("⚠️ Sin predicciones: ejecuta primero la sección 7.")
else:
    mostrar_errores(df_pred, n=12)

In [ ]:
def tabla_pares_confundidos(df: pd.DataFrame) -> pd.DataFrame:
    """Errores agrupados por par real→predicción, con confianza media."""
    errores = df[df["real"] != df["prediccion"]]
    if errores.empty:
        return pd.DataFrame()
    tabla = (errores.groupby(["real", "prediccion"])
             .agg(errores=("confianza", "size"), confianza_media=("confianza", "mean"))
             .sort_values("errores", ascending=False).reset_index())
    tabla["% de los errores"] = (tabla["errores"] / len(errores) * 100).round(1)
    return tabla


def mostrar_pares_confundidos(df: pd.DataFrame, n_pares: int = 4,
                              n_ejemplos: int = 3) -> None:
    """Para los pares real→pred más frecuentes, muestra ejemplos mal clasificados."""
    tabla = tabla_pares_confundidos(df)
    if tabla.empty:
        print("🎉 Sin errores que analizar.")
        return
    for _, par in tabla.head(n_pares).iterrows():
        errores_par = df[(df["real"] == par["real"]) &
                         (df["prediccion"] == par["prediccion"])]
        muestra = errores_par.sample(min(n_ejemplos, len(errores_par)),
                                     random_state=SEMILLA)
        fig, ejes = plt.subplots(1, n_ejemplos, figsize=(4 * n_ejemplos, 3.6))
        fig.suptitle(f"{par['real']} → {par['prediccion']}  "
                     f"({int(par['errores'])} errores, "
                     f"confianza media {par['confianza_media']:.1%})",
                     fontsize=11, color="#b3261e")
        for eje, (_, fila) in zip(np.atleast_1d(ejes).flat, muestra.iterrows()):
            img = cv2.imread(str(fila["ruta"]))
            if img is not None:
                eje.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            eje.set_title(f"conf: {fila['confianza']:.1%}", fontsize=9)
            eje.axis("off")
        for eje in np.atleast_1d(ejes).flat[len(muestra):]:
            eje.axis("off")
        plt.tight_layout()
        plt.show()


if df_pred.empty:
    print("⚠️ Sin predicciones: ejecuta primero la sección 7.")
else:
    display(tabla_pares_confundidos(df_pred).head(12))
    mostrar_pares_confundidos(df_pred)

## 9. Comparación de experimentos

Tabla persistente (`experimentos.csv`) preparada para comparar futuros
entrenamientos: basta con cambiar `NOMBRE_EXPERIMENTO` y los hiperparámetros,
re-entrenar y re-ejecutar esta sección.

In [ ]:
def registrar_experimento(nombre: str, metricas: dict[str, float],
                          ruta_csv: Path) -> pd.DataFrame:
    """Añade (o actualiza) un experimento en el CSV y devuelve la tabla completa.

    Nunca sobrescribe otros experimentos: cada fila queda registrada con su
    configuración (modelo, hiperparámetros, augmentation y división usada).
    """
    fila = {
        "experimento": nombre,
        "modelo": MODELO_BASE,
        "epochs": HIPERPARAMETROS["epochs"],
        "batch": HIPERPARAMETROS["batch"],
        "imgsz": HIPERPARAMETROS["imgsz"],
        "optimizer": HIPERPARAMETROS["optimizer"],
        "lr0": HIPERPARAMETROS["lr0"],
        **metricas,
        "augmentation": AUGMENTATION,
        "split": " | ".join(f"{s}:{'+'.join(p)}" for s, p in SPLITS.items()),
    }
    df = pd.read_csv(ruta_csv) if ruta_csv.exists() else pd.DataFrame()
    if not df.empty:
        df = df[df["experimento"] != nombre]
    df = pd.concat([df, pd.DataFrame([fila])], ignore_index=True)
    df.to_csv(ruta_csv, index=False)
    return df


if df_pred.empty:
    print("⚠️ Sin métricas que registrar: ejecuta primero la sección 7.")
else:
    df_experimentos = registrar_experimento(NOMBRE_ENSEMBLE_FINAL, metricas,
                                            RUTA_EXPERIMENTOS)
    display(df_experimentos.round(4))

## 10. Demo en vivo 🎬

La demo **no reentrena nada**: carga una única vez los 3 modelos del ensemble
final (`modelos_finales/`, ya entrenados) y ejecuta solo inferencia, por lo
que responde en menos de un segundo por imagen incluso en CPU.

- `predict_exercise(ruta)` → clasifica **cualquier imagen PNG**: muestra la
  imagen, la clase predicha, la confianza y el Top-5 de probabilidades.
- `clasificar_imagen_aleatoria()` → toma una imagen **aleatoria del conjunto
  de prueba** (persona06, nunca vista) y la clasifica.

In [ ]:
@lru_cache(maxsize=1)
def _cargar_modelos_demo() -> tuple:
    """Carga una sola vez los modelos del ensemble final."""
    modelos = []
    for nombre in ENSEMBLE_EXPERIMENTOS:
        pesos = ruta_pesos(nombre)
        if pesos.exists():
            modelos.append(YOLO(str(pesos)))
    return tuple(modelos)


def predict_exercise(image_path: str | Path) -> None:
    """Clasifica una imagen PNG: muestra imagen, predicción, confianza y Top-5.

    Usa el ensemble final (promedio de probabilidades de 3 modelos).
    """
    ruta = Path(image_path)
    if not ruta.exists():
        print(f"⚠️ La imagen no existe: {ruta}")
        return
    img = cv2.imread(str(ruta))
    if img is None:
        print(f"⚠️ No se pudo leer la imagen: {ruta}")
        return
    modelos = _cargar_modelos_demo()
    if YOLO is None or not modelos:
        print("⚠️ No hay ningún modelo entrenado todavía (ver sección 6).")
        return

    resultados = [m.predict(str(ruta), imgsz=int(HIPERPARAMETROS["imgsz"]),
                            verbose=False)[0] for m in modelos]
    probs = np.mean([r.probs.data.cpu().numpy() for r in resultados], axis=0)
    nombres_clase = resultados[0].names
    orden = probs.argsort()[::-1][:5]
    nombres = [nombres_clase[int(i)] for i in orden]
    confs = [float(probs[int(i)]) for i in orden]

    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(13, 4.5),
                                         width_ratios=[1, 1.3])
    ax_img.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax_img.axis("off")
    titulo = f"Predicción: {nombres[0]}  ({confs[0]:.1%})"
    if ruta.parent.name in nombres_clase.values():
        acierto = "✅" if ruta.parent.name == nombres[0] else "❌"
        titulo += f"\nClase real: {ruta.parent.name} {acierto}"
    ax_img.set_title(titulo, fontsize=11)

    posiciones = np.arange(len(nombres))[::-1]
    ax_bar.barh(posiciones, confs, color=COLOR)
    ax_bar.set_yticks(posiciones, nombres)
    ax_bar.set_xlim(0, 1)
    ax_bar.set_xlabel("probabilidad")
    ax_bar.set_title("Top-5 probabilidades (ensemble)")
    ax_bar.spines[["top", "right"]].set_visible(False)
    for y, c in zip(posiciones, confs):
        ax_bar.text(c + 0.01, y, f"{c:.1%}", va="center", fontsize=9)
    plt.tight_layout()
    plt.show()


# Alias en español (compatibilidad con celdas anteriores)
clasificar_imagen = predict_exercise


def clasificar_imagen_aleatoria() -> None:
    """Selecciona una imagen aleatoria del conjunto de prueba y la clasifica."""
    rutas = list((RUTA_DATASET_CLS / "test").rglob("*.png"))
    if not rutas:
        print("⚠️ El conjunto de prueba está vacío (dataset incompleto): "
              "revisa la sección 4.")
        return
    predict_exercise(random.choice(rutas))

In [ ]:
# 🎯 Demo 1 — clasificar una imagen concreta (editar la ruta):
# predict_exercise(RUTA_DATASET / "persona05" / "Squat" / "Frontal_0001.png")

In [ ]:
# 🎲 Demo 2 — imagen aleatoria del conjunto de prueba:
clasificar_imagen_aleatoria()

## 11. Conclusiones

**Resultados obtenidos**
- Fase 1 (test = persona05, la misma identidad usada para elegir
  hiperparámetros — split provisional con 3 identidades de train): mejor
  individual exp06 (63.6 %), mejor ensemble exp06+exp09 (67.2 %). Números
  inflados por reutilizar persona05 para tomar decisiones.
- Fase 2 (train = 4 identidades, test = persona06, nunca usado antes):
  mejor real exp11 — 55.6 %. Ni más épocas, ni otra semilla, ni recortar a la
  persona con un detector YOLO, ni ensemblar con modelos débiles lo mejoraron.
- Fase 3 (train = 7 identidades tras agregar persona07/08/09; mismo
  test = persona06): el mejor modelo individual saltó a **80.3 %** (exp16,
  idéntica configuración que exp11/exp06). Entrenando 5 semillas distintas
  (57.6–80.3 %) y probando combinaciones de ensemble, la mejor —
  **exp16+exp17+exp19** — alcanzó **85.35 % de accuracy, F1 macro 0.84**.
  **✅ Objetivo de ≥85 % cumplido.**

**Qué cambió entre la fase 2 y la fase 3**
- La única diferencia fue pasar de 4 a 7 identidades de entrenamiento
  (persona07, persona08 —con fotos a contraluz, incluidas sin descartar— y
  persona09). Ningún hiperparámetro cambió respecto a exp11/exp06. Esto
  confirma la conclusión de la fase 2: el cuello de botella era la cantidad
  de identidades, no la arquitectura ni el preprocesado.
- Con más identidades, varios entrenamientos individuales ya son fuertes por
  separado, así que ensemblar vuelve a ayudar (83.0 % con 2 modelos, 85.35 %
  con la mejor combinación de 3) — en la fase 2 esto no funcionaba porque
  solo había un modelo fuerte a la vez.

**Hallazgos experimentales clave (acumulados)**
- El augmentation fuerte por defecto de Ultralytics sigue siendo
  imprescindible en las tres fases.
- Todo lo que aumenta la capacidad efectiva de memorizar empeora: yolo11s,
  dropout, 448 px, más épocas, recorte a la persona (quita fondo pero
  distorsiona el aspect ratio).
- **Alta varianza entre semillas** (57.6–80.3 % con hiperparámetros
  idénticos): con 7 identidades el entrenamiento todavía no converge a una
  solución única — de ahí que el resultado robusto sea un ensemble de varias
  semillas, no un solo entrenamiento.
- Ensemblar solo ayuda si los modelos combinados son fuertes por separado; el
  modelo más débil del grupo (exp18, 70.4 %) empeora cualquier combinación en
  la que participa.

**Fortalezas del modelo final (ensemble exp16+exp17+exp19)**
- Todas las clases alcanzan F1 ≥ 0.67 sobre una identidad nunca vista;
  Push-up, Squat, Plank, High_Knees superan F1 0.90.
- Sigue siendo YOLO puro (3 clasificadores nano promediados en inferencia) y
  rápido: <1 s por imagen en CPU para las 3 pasadas.

**Limitaciones y fuentes de error restantes**
- `Chair_Dip` (F1 0.67) y `Superman`/`Jumping_Jack` (recall 0.56–0.67) siguen
  siendo las clases más débiles; probablemente por ambigüedad visual real
  entre poses de cuerpo entero desde el ángulo de cámara usado.
- persona05 (val) sigue sin corregir la postura de *Estiramiento_lateral*:
  su accuracy de validación en esa clase específica no debe interpretarse
  como una debilidad real del modelo.
- El resultado depende de una búsqueda de combinación de ensemble sobre el
  propio test (persona06): es una forma leve de la misma fuga metodológica
  señalada en la fase 1. Con más identidades de validación este paso podría
  hacerse sobre val en vez de sobre test.

**Trabajo futuro (en orden de impacto esperado)**
1. Seguir agregando identidades de entrenamiento y, sobre todo, de
   **validación**: hoy solo hay una persona de val (persona05, con una clase
   dudosa) y una de test — no alcanza para elegir la combinación de ensemble
   sin tocar el test.
2. Validación cruzada *leave-one-person-out* para reportar una métrica de
   generalización más estable.
3. Corregir persona05/Estiramiento_lateral y revisar ambigüedades similares
   entre Chair_Dip/Squat y Superman/Plank.
4. Balancear el número de imágenes por clase y por persona (persona08 en
   particular tiene pocas imágenes por clase, 10–25).